# CRC cell fractions by consensus molecular subtype

Load plotting settings and the shared CMS order and colors for CRC1, CRC2, and CRC3.

In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
from scipy.stats import mannwhitneyu
import matplotlib as mpl
import numpy as np
from matplotlib.patches import Patch

warnings.filterwarnings("ignore")
mpl.rcParams.update(
    {
        "font.family": "sans-serif",
        "font.sans-serif": ["DejaVu Sans"],
        "font.size": 8,
        "axes.titlesize": 9,
        "axes.labelsize": 8,
        "xtick.labelsize": 7,
        "ytick.labelsize": 7,
        "legend.fontsize": 7,
        "axes.linewidth": 0.7,
        "xtick.major.width": 0.7,
        "ytick.major.width": 0.7,
        "xtick.major.size": 3,
        "ytick.major.size": 3,
        "xtick.direction": "out",
        "ytick.direction": "out",
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "savefig.facecolor": "white",
        "savefig.edgecolor": "white",
        "savefig.dpi": 300,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    }
)
methods = ["SABER", "DWLS", "BayesPrism"]
npg_colors = ["#3C5488", "#E64B35", "#00A087", "#4DBBD5", "#F39B7F"]
cms_order = ["CMS1", "CMS2", "CMS3", "CMS4"]
cms_palette = dict(zip(cms_order, npg_colors))


## Compare predicted fractions across CMS groups

For SABER, DWLS, and BayesPrism, match CMS labels to predictions, use samples as statistical units, compare the highest-mean CMS group with the other groups, and save one six-panel PDF per method.


In [5]:
cohort = "CRC"
output_folder = "plots_cms_boxplots"
cms_column = "RF.nearestCMS"

plots_dir = f"/Path/to/output/cancer/CRC/{output_folder}"
prediction_paths = {method: f"/Path/to/cancer/results/{cohort}/{method}.csv" for method in methods}
missing_prediction_files = [path for path in prediction_paths.values() if not os.path.exists(path)]
if missing_prediction_files:
    raise FileNotFoundError(f"Missing required prediction files: {missing_prediction_files}")

os.makedirs(plots_dir, exist_ok=True)
cms_df = pd.read_csv(f"/Path/to/cancer/data/{cohort}/CMS_result.csv", index_col=0)
cms_df.index = cms_df.index.str.replace(".", "-", regex=False)
cms_labels = cms_df[cms_column].dropna()

for method in methods:
    prediction_df = pd.read_csv(prediction_paths[method], index_col=0)
    prediction_df.index = prediction_df.index.str.replace(".", "-", regex=False)
    common_samples = cms_labels.index.intersection(prediction_df.index)
    plot_data_dict = {}
    for cell_type in [
        "Fibroblasts", "Endothelial", "NK_cells", "T_cells_Cytotoxic",
        "Macrophages", "Smooth_Muscle",
    ]:
        plot_data = prediction_df.loc[common_samples, [cell_type]].copy()
        plot_data["CMS"] = cms_labels.loc[common_samples]
        plot_data = plot_data[plot_data["CMS"].isin(cms_order)]
        plot_data["CMS"] = pd.Categorical(
            plot_data["CMS"], categories=cms_order, ordered=True
        )
        plot_data_dict[cell_type] = plot_data

    fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(7.0, 4.8))
    axes = np.array(axes).flatten()
    for ax, (cell_type, plot_data) in zip(axes, plot_data_dict.items()):
        sns.boxplot(
            x="CMS", y=cell_type, data=plot_data, order=cms_order,
            palette=cms_palette, ax=ax, width=0.58, linewidth=0.8,
            fliersize=0, saturation=0.95,
            boxprops=dict(edgecolor="black", linewidth=0.8),
            whiskerprops=dict(color="black", linewidth=0.8),
            capprops=dict(color="black", linewidth=0.8),
            medianprops=dict(color="black", linewidth=1.0),
        )
        sns.stripplot(
            x="CMS", y=cell_type, data=plot_data, order=cms_order,
            ax=ax, color="black", size=2.0, alpha=0.35,
            jitter=0.22, linewidth=0,
        )
        ax.set_title(cell_type, fontsize=9, pad=5, fontweight="normal")
        ax.set_xlabel("")
        ax.set_ylabel("Proportion", fontsize=8)
        max_cms = plot_data.groupby("CMS", observed=False)[cell_type].mean().idxmax()
        data_min = plot_data[cell_type].min()
        data_max_value = plot_data[cell_type].max()
        y_range = data_max_value - data_min
        assert y_range != 0
        h = y_range * 0.05
        line_y = data_max_value + h
        final_y = line_y
        reference_values = plot_data.loc[plot_data["CMS"] == max_cms, cell_type]
        for comparison_cms in cms_order:
            if comparison_cms == max_cms:
                continue
            comparison_values = plot_data.loc[plot_data["CMS"] == comparison_cms, cell_type]
            assert len(reference_values) >= 2 and len(comparison_values) >= 2
            _, p_value = mannwhitneyu(
                reference_values, comparison_values, alternative="two-sided"
            )
            significance = (
                "***" if p_value < 0.001 else
                "**" if p_value < 0.01 else
                "*" if p_value < 0.05 else None
            )
            if significance:
                x_reference = cms_order.index(max_cms)
                x_comparison = cms_order.index(comparison_cms)
                ax.plot(
                    [x_reference, x_reference, x_comparison, x_comparison],
                    [line_y, line_y + h / 2, line_y + h / 2, line_y],
                    linewidth=0.8,
                    color="black",
                )
                ax.text(
                    (x_reference + x_comparison) / 2,
                    line_y + h / 2,
                    significance,
                    ha="center",
                    va="bottom",
                    fontsize=8,
                    color="black",
                )
                final_y = line_y + h / 2
                line_y += h * 1.6
        ax.set_ylim(data_min - y_range * 0.08, final_y + y_range * 0.2)
        ax.tick_params(axis="x", which="both", labelbottom=False)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.spines["left"].set_linewidth(0.7)
        ax.spines["bottom"].set_linewidth(0.7)
        ax.tick_params(axis="both", which="major", width=0.7, length=3, pad=2)
        ax.grid(False)

    cms_counts = next(iter(plot_data_dict.values()))["CMS"].value_counts()
    fig.legend(
        handles=[
            Patch(
                facecolor=cms_palette[cms],
                edgecolor="black",
                linewidth=0.5,
                label=f"{cms} (n={cms_counts.get(cms, 0)})",
            )
            for cms in cms_order
        ],
        loc="lower center",
        ncol=len(cms_order),
        frameon=False,
        bbox_to_anchor=(0.5, 0.01),
        handlelength=1.1,
        handleheight=0.8,
        handletextpad=0.5,
        columnspacing=1.2,
        fontsize=7,
    )
    fig.suptitle(
        f"{method} - Cell Type Proportion by CMS",
        fontsize=12,
        fontweight="normal",
        y=0.995,
    )
    plt.tight_layout(rect=[0, 0.08, 1, 0.96])
    plt.savefig(
        f"{plots_dir}/{method}_CMS_boxplot_max_vs_others.pdf",
        bbox_inches="tight",
    )
    plt.close()
